# 📚 Task 1 — Web Scraping: books.toscrape.com

**Target:** https://books.toscrape.com/catalogue/page-1.html

**Extracting:**
- 📖 Book Name
- 📝 Book Description *(from individual book page)*
- 💰 Book Price
- 📦 Stock Availability
- 📄 Page Number

**Methods:** Requests + BeautifulSoup &nbsp;|&nbsp; Scrapy

---

## ⚙️ Install Required Libraries

In [ ]:
%pip install requests beautifulsoup4 lxml pandas scrapy -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.7/331.7 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.9/264.9 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 3.9 MB/s eta 0:00:00


---
# 🥣 Method 1: Requests + BeautifulSoup


### Cell 1 — Imports

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

### Cell 2 — Scrape ONE book page (test)

In [ ]:
BASE_URL = "https://books.toscrape.com/catalogue/"
HEADERS  = {"User-Agent": "Mozilla/5.0"}

# نجرب صفحة 1 بس أول ما نشوف إن الكود شغال
url      = BASE_URL + "page-1.html"
response = requests.get(url, headers=HEADERS)
soup     = BeautifulSoup(response.text, "lxml")

books = soup.find_all("article", class_="product_pod")
print(f"✅ Found {len(books)} books on page 1")

# نطبع أول كتاب كـ test
first = books[0]
print("Name  :", first.h3.a["title"])
print("Price :", first.find("p", class_="price_color").get_text(strip=True))
print("Stock :", first.find("p", class_="instock availability").get_text(strip=True))

✅ Found 20 books on page 1
Name  : A Light in the Attic
Price : Â£51.77
Stock : In stock


### Cell 3 — Get Description from individual book page (test)

In [ ]:
# نجرب نجيب الـ description من صفحة أول كتاب
relative_url = books[0].h3.a["href"].replace("../../", "")
book_url     = BASE_URL + relative_url

book_resp = requests.get(book_url, headers=HEADERS)
book_soup = BeautifulSoup(book_resp.text, "lxml")

desc_tag    = book_soup.find("div", id="product_description")
description = desc_tag.find_next_sibling("p").get_text(strip=True) if desc_tag else "No description"

print("Book URL   :", book_url)
print("Description:", description[:150], "...")

Book URL   : https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
Description: It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20t ...


### Cell 4 — Scrape ALL 50 pages 🚀

In [ ]:
BASE_URL = "https://books.toscrape.com/catalogue/"
HEADERS  = {"User-Agent": "Mozilla/5.0"}

all_books = []

for page_num in range(1, 51):

    # ── بناء الـ URL أوتوماتيك لكل صفحة ──────────────────────────────────
    url      = f"{BASE_URL}page-{page_num}.html"
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        print(f"⚠️ Page {page_num} not found, stopping.")
        break

    soup  = BeautifulSoup(response.text, "lxml")
    books = soup.find_all("article", class_="product_pod")
    print(f"📄 Page {page_num} → {len(books)} books")

    for book in books:

        # ── 1) اسم الكتاب ────────────────────────────────────────────────
        name  = book.h3.a["title"]

        # ── 2) السعر ─────────────────────────────────────────────────────
        price = book.find("p", class_="price_color").get_text(strip=True)

        # ── 3) الـ Stock ──────────────────────────────────────────────────
        stock = book.find("p", class_="instock availability").get_text(strip=True)

        # ── 4) ادخل صفحة الكتاب وجيب الـ description ────────────────────
        relative_url = book.h3.a["href"].replace("../../", "")
        book_url     = BASE_URL + relative_url

        book_resp = requests.get(book_url, headers=HEADERS)
        book_soup = BeautifulSoup(book_resp.text, "lxml")

        desc_tag    = book_soup.find("div", id="product_description")
        description = desc_tag.find_next_sibling("p").get_text(strip=True) if desc_tag else "No description"

        # ── إضافة الكتاب للقائمة ─────────────────────────────────────────
        all_books.append({
            "book_name":        name,
            "book_description": description,
            "book_price":       price,
            "stock":            stock,
            "page":             page_num
        })

print(f"\n✅ Total books scraped: {len(all_books)}")

📄 Page 1 → 20 books
📄 Page 2 → 20 books
📄 Page 3 → 20 books
📄 Page 4 → 20 books
📄 Page 5 → 20 books
📄 Page 6 → 20 books
📄 Page 7 → 20 books
📄 Page 8 → 20 books
📄 Page 9 → 20 books
📄 Page 10 → 20 books
📄 Page 11 → 20 books
📄 Page 12 → 20 books
📄 Page 13 → 20 books
📄 Page 14 → 20 books
📄 Page 15 → 20 books
📄 Page 16 → 20 books
📄 Page 17 → 20 books
📄 Page 18 → 20 books
📄 Page 19 → 20 books
📄 Page 20 → 20 books
📄 Page 21 → 20 books
📄 Page 22 → 20 books
📄 Page 23 → 20 books
📄 Page 24 → 20 books
📄 Page 25 → 20 books
📄 Page 26 → 20 books
📄 Page 27 → 20 books
📄 Page 28 → 20 books
📄 Page 29 → 20 books
📄 Page 30 → 20 books
📄 Page 31 → 20 books
📄 Page 32 → 20 books
📄 Page 33 → 20 books
📄 Page 34 → 20 books
📄 Page 35 → 20 books
📄 Page 36 → 20 books
📄 Page 37 → 20 books
📄 Page 38 → 20 books
📄 Page 39 → 20 books
📄 Page 40 → 20 books
📄 Page 41 → 20 books
📄 Page 42 → 20 books
📄 Page 43 → 20 books
📄 Page 44 → 20 books
📄 Page 45 → 20 books
📄 Page 46 → 20 books
📄 Page 47 → 20 books
📄 Page 48 → 20 books
📄

### Cell 5 — Convert to DataFrame & Preview

In [ ]:
df_bs4 = pd.DataFrame(all_books)
print(f"Shape: {df_bs4.shape}")
df_bs4.head(10)

Shape: (1000, 5)


,book_name,book_description,book_price,stock,page
0,A Light in the Attic,It's hard to imagine a world without A Light i...,Â£51.77,In stock,1
1,Tipping the Velvet,"""Erotic and absorbing...Written with starling ...",Â£53.74,In stock,1
2,Soumission,"Dans une France assez proche de la nÃ´tre, un ...",Â£50.10,In stock,1
3,Sharp Objects,"WICKED above her hipbone, GIRL across her hear...",Â£47.82,In stock,1
4,Sapiens: A Brief History of Humankind,From a renowned historian comes a groundbreaki...,Â£54.23,In stock,1
5,The Requiem Red,Patient Twenty-nine.A monster roams the halls ...,Â£22.65,In stock,1
6,The Dirty Little Secrets of Getting Your Dream...,Drawing on his extensive experience evaluating...,Â£33.34,In stock,1
7,The Coming Woman: A Novel Based on the Life of...,"""If you have a heart, if you have a soul, Kare...",Â£17.93,In stock,1
8,The Boys in the Boat: Nine Americans and Their...,For readers of Laura Hillenbrand's Seabiscuit ...,Â£22.60,In stock,1
9,The Black Maria,"Praise for Aracelis Girmay:""[Girmay's] every l...",Â£52.15,In stock,1


### Cell 6 — Save to CSV & Download

In [ ]:
df_bs4.to_csv("books_bs4.csv", index=False, encoding="utf-8")
print("💾 Saved: books_bs4.csv")

💾 Saved: books_bs4.csv


---
# 🕷️ Method 2: Scrapy

الـ Scrapy مش بيشتغل جوه الـ notebook مباشرة زي الـ BS4،  
بس عندنا حل: بنكتب ملفات الـ Spider بـ `%%writefile` وبعدين نشغله بـ `!scrapy crawl`

### Cell 7 — Create Scrapy Project

In [ ]:
!scrapy startproject books_project
%cd books_project
print("✅ Project created!")

New Scrapy project 'books_project', using template directory '/usr/local/lib/python3.12/dist-packages/scrapy/templates/project', created in:
    /content/books_project

You can start your first spider with:
    cd books_project
    scrapy genspider example example.com
/content/books_project
✅ Project created!


### Cell 8 — Write the Spider File

In [ ]:
import os

spider_code = """
import scrapy

class BooksSpider(scrapy.Spider):

    # ── اسم الـ Spider ────────────────────────────────────────────────────
    name = "books"

    # ── الصفحة الأولى ────────────────────────────────────────────────────
    start_urls = ["https://books.toscrape.com/catalogue/page-1.html"]

    current_page = 1

    # ════════════════════════════════════════════════════════════════════
    # parse() — بتشتغل على كل صفحة قائمة كتب
    # ════════════════════════════════════════════════════════════════════
    def parse(self, response):

        books = response.css("article.product_pod")

        for book in books:

            # ── اسم الكتاب ───────────────────────────────────────────────
            name  = book.css("h3 a::attr(title)").get()

            # ── السعر ────────────────────────────────────────────────────
            price = book.css("p.price_color::text").get()

            # ── الـ Stock ─────────────────────────────────────────────────
            stock = " ".join(book.css("p.instock.availability::text").getall()).strip()

            # ── رابط صفحة الكتاب ─────────────────────────────────────────
            book_url = response.urljoin(book.css("h3 a::attr(href)").get())

            # ── روح صفحة الكتاب عشان الـ description ─────────────────────
            yield scrapy.Request(
                url      = book_url,
                callback = self.parse_book,
                meta     = {
                    "book_name":  name,
                    "book_price": price,
                    "stock":      stock,
                    "page":       self.current_page,
                }
            )

        # ── Pagination: روح الصفحة الجاية أوتوماتيك ──────────────────────
        next_page = response.css("li.next a::attr(href)").get()
        if next_page:
            self.current_page += 1
            yield scrapy.Request(
                url      = response.urljoin(next_page),
                callback = self.parse
            )

    # ════════════════════════════════════════════════════════════════════
    # parse_book() — بتشتغل على صفحة كل كتاب منفرد
    # ════════════════════════════════════════════════════════════════════
    def parse_book(self, response):

        description = response.css("div#product_description + p::text").get(
            default="No description"
        )

        yield {
            "book_name":        response.meta["book_name"],
            "book_description": description,
            "book_price":       response.meta["book_price"],
            "stock":            response.meta["stock"],
            "page":             response.meta["page"],
        }
"""

# تحديد المسار الذي نريد الحفظ فيه
folder_path = "books_project/spiders"
file_path = f"{folder_path}/books_spider.py"

# إنشاء المجلدات إذا لم تكن موجودة بالفعل
os.makedirs(folder_path, exist_ok=True)

# حفظ الملف بترميز utf-8 لضمان عدم حدوث مشاكل مع النصوص
with open(file_path, "w", encoding="utf-8") as f:
    f.write(spider_code)

print(f"✅ Spider file created and saved successfully at: {file_path}")

✅ Spider file created and saved successfully at: books_project/spiders/books_spider.py


### Cell 9 — Run Spider & Export CSV 🚀

In [ ]:
!scrapy crawl books -o books_scrapy.csv --nolog
print("\n✅ Done! books_scrapy.csv created")


✅ Done! books_scrapy.csv created


### Cell 10 — Preview Results

In [ ]:
%pip install scrapy

In [ ]:
!python -m scrapy runspider books_project/spiders/books_spider.py -O books_scrapy.csv

In [ ]:
import pandas as pd

df_scrapy = pd.read_csv("books_scrapy.csv")
print(f"Shape: {df_scrapy.shape}")
df_scrapy.head(10)

### Cell 11 — Download CSV

In [ ]:
import os

file_path = os.path.join(os.getcwd(), "books_scrapy.csv")
print("📂 ملفك موجود هنا  يسطى ركز:")
print(file_path)